# Kaggle Training: LogFilter NER on CyNER Corpus

Fine-tunes `cisco-ai/SecureBERT2.0-NER` on **CyNER** — a real, hand-labelled cybersecurity NER corpus (arXiv:2204.05754, Alam et al., 2022; MIT licence). CyNER covers the same five entity families the production wrapper exposes: `Indicator`, `Malware`, `Organization`, `System`, `Vulnerability`.

**Domain-transfer caveat.** CyNER is drawn from CTI prose (threat reports, MITRE ATT&CK summaries); production logs use syslog syntax where the same entities appear in different surface forms (e.g. `src: /10.251.74.62:35977` instead of `the actor used 10.251.74.62`). Fine-tuning on CyNER improves base recall but requires validation against held-out syslog windows before promotion. See Section 9 for the promotion gate.

Output: `models/ner/` — HuggingFace token-classification directory drop-in for `src/logfilter/models/ner.py`. Expected runtime on Kaggle Free GPU (T4) is well under the 9h session limit. Enable a GPU accelerator before running training cells.


## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()
GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]
    else:
        clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
        if not clone_dir.exists():
            subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
        REPO_DIR = clone_dir

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## 2. Install training dependencies

Adds `seqeval` for token-level F1 (including per-class reporting), plus the standard transformers/datasets/optimum stack.


In [ ]:
%pip install -q transformers datasets accelerate seqeval evaluate optimum[onnxruntime] onnx onnxruntime


## 3. Load CyNER corpus

Clones the CyNER repository (MIT licence, arXiv:2204.05754) and parses the `dataset/mitre/{train,valid,test}.txt` files. The corpus uses a 2-column CoNLL format (token TAB BIO-tag) with blank lines as sentence boundaries. All 11 tags match the project's BIO scheme exactly — no label remapping needed.

**Corpus size:** train 68,191 tokens / 2,811 sentences; valid 19,530 / 813; test 19,270 / 748. Note that `B-Vulnerability` has only ~67 occurrences across all splits — per-class F1 for this class should be monitored closely.


In [ ]:
from collections import Counter

CYNER_DIR = Path('/kaggle/working/CyNER')
CYNER_COMMIT = '37aff53bd7235605638320e0c06de2fb82847070'
CYNER_URL = 'https://github.com/aiforsec/CyNER'

# Clone CyNER if not already present (mirrors self-bootstrap pattern from Section 1)
if not CYNER_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', CYNER_URL, str(CYNER_DIR)],
        check=True,
    )
    print('Cloned CyNER to:', CYNER_DIR)
else:
    print('CyNER directory already present:', CYNER_DIR)

# Pin to reproducible commit
try:
    subprocess.run(
        ['git', 'checkout', CYNER_COMMIT],
        check=True, cwd=str(CYNER_DIR), capture_output=True,
    )
except subprocess.CalledProcessError:
    # depth-1 clone may not have the pinned commit; fetch it explicitly
    subprocess.run(
        ['git', 'fetch', '--depth', '1', 'origin', CYNER_COMMIT],
        check=True, cwd=str(CYNER_DIR),
    )
    subprocess.run(['git', 'checkout', CYNER_COMMIT], check=True, cwd=str(CYNER_DIR))
print('Pinned to commit:', CYNER_COMMIT)

# Valid BIO tags -- matches the project's 11-tag scheme exactly (no remapping needed)
VALID_TAGS = {
    'O',
    'B-Indicator', 'I-Indicator',
    'B-Malware', 'I-Malware',
    'B-Organization', 'I-Organization',
    'B-System', 'I-System',
    'B-Vulnerability', 'I-Vulnerability',
}


def parse_conll(path: Path) -> list[dict]:
    """Parse CyNER CoNLL file: token<TAB>tag per line, blank line = sentence boundary."""
    sentences = []
    tokens: list[str] = []
    tags: list[str] = []
    for lineno, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            if tokens:
                sentences.append({'tokens': tokens, 'tags': tags})
                tokens, tags = [], []
        else:
            parts = line.rstrip('\n').split('\t')
            if len(parts) != 2:
                raise ValueError(
                    f'Expected 2 tab-separated columns at line {lineno} in {path}: {line!r}'
                )
            token, tag = parts
            if tag not in VALID_TAGS:
                raise ValueError(
                    f'Unknown tag {tag!r} at line {lineno} in {path} (token={token!r}). '
                    f'Expected one of: {sorted(VALID_TAGS)}'
                )
            tokens.append(token)
            tags.append(tag)
    if tokens:  # last sentence without trailing blank line
        sentences.append({'tokens': tokens, 'tags': tags})
    return sentences


# Load all three splits from dataset/mitre/
data: dict[str, list[dict]] = {}
for split in ('train', 'valid', 'test'):
    corpus_path = CYNER_DIR / 'dataset' / 'mitre' / f'{split}.txt'
    if not corpus_path.exists():
        raise FileNotFoundError(f'CyNER corpus file not found: {corpus_path}')
    data[split] = parse_conll(corpus_path)
    n_tokens = sum(len(s['tokens']) for s in data[split])
    print(f'{split}: {len(data[split])} sentences, {n_tokens} tokens')

# Per-class B-* counts -- flags minority classes (Vulnerability has ~67 B-* tokens)
b_counts: Counter = Counter()
for sents in data.values():
    for sent in sents:
        for tag in sent['tags']:
            if tag.startswith('B-'):
                b_counts[tag] += 1

print('\nB-* entity counts (all splits combined):')
for tag, count in sorted(b_counts.items(), key=lambda x: -x[1]):
    flag = '  <- LOW (<100): monitor F1 closely' if count < 100 else ''
    print(f'  {tag}: {count}{flag}')


## 4. Tokenise and align labels

Defines the 11-tag BIO label scheme, validates a synthetic fixture from the CyNER spec, and provides `tokenize_and_align_labels()` for subword-aware label projection. CyNER data is already word-tokenized, so `is_split_into_words=True` is used. The first subword of each word gets the original BIO label; subsequent subwords get the corresponding `I-` variant; special tokens get `-100` (ignored by the loss).

`max_length=256` is safe for CyNER sentences after subword tokenization; 128 is risky.


In [ ]:
LABEL_LIST = [
    'O',
    'B-Indicator', 'I-Indicator',
    'B-Malware', 'I-Malware',
    'B-Organization', 'I-Organization',
    'B-System', 'I-System',
    'B-Vulnerability', 'I-Vulnerability',
]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

MAX_LENGTH = 256  # safe for CyNER after subword tokenization; 128 is risky

# Synthetic 10-token fixture from the CyNER spec -- validates label projection
FIXTURE_TOKENS = [
    'Super', 'Mario', 'Run', 'Malware', '#', '2', '\u2013', 'DroidJack', 'RAT', 'Gamers',
]
FIXTURE_TAGS = [
    'B-Malware', 'I-Malware', 'I-Malware', 'I-Malware', 'O', 'O', 'O',
    'B-Malware', 'I-Malware', 'O',
]
assert len(FIXTURE_TOKENS) == len(FIXTURE_TAGS), 'Fixture length mismatch'
assert all(t in LABEL2ID for t in FIXTURE_TAGS), (
    f'Unknown fixture tag: {[t for t in FIXTURE_TAGS if t not in LABEL2ID]}'
)
print('Fixture label IDs:', [LABEL2ID[t] for t in FIXTURE_TAGS])
# Expected: [3, 4, 4, 4, 0, 0, 0, 3, 4, 0]  (B-Malware=3, I-Malware=4, O=0)


def tokenize_and_align_labels(examples: list[dict], tokenizer) -> list[dict]:
    """
    Tokenise pre-tokenized CyNER sentences with subword label alignment.

    CyNER data is already word-tokenized (one word per token), so we use
    is_split_into_words=True.  BIO scheme is preserved: the first subword of
    a word gets the original label; subsequent subwords get the I- variant
    (or -100 for special tokens).
    """
    encoded = []
    for ex in examples:
        tokens = ex['tokens']
        word_tags = ex['tags']
        enc = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        word_ids = enc.word_ids()
        labels = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)
            elif word_id != prev_word_id:
                labels.append(LABEL2ID[word_tags[word_id]])
            else:
                tag = word_tags[word_id]
                if tag.startswith('B-'):
                    tag = 'I-' + tag[2:]
                labels.append(LABEL2ID[tag])
            prev_word_id = word_id
        enc['labels'] = labels
        encoded.append(dict(enc))
    return encoded


In [ ]:
# OPTIONAL: regex weak-supervision augmentation (legacy path)
# -----------------------------------------------------------
# Preserves the original HDFS-TraceBench weak-supervision approach.
# NOT part of the default CyNER training path above.
# To activate: set USE_REGEX_AUGMENTATION = True, run this cell before
# Section 5, then merge regex_encoded into the CyNER training data.
#
# USE_REGEX_AUGMENTATION = False
#
# import re
# from training.text_dataset import build_windows
#
# PATTERNS = [
#     (re.compile(r'\bCVE-\d{4}-\d{4,7}\b', re.IGNORECASE), 'Vulnerability'),
#     (re.compile(r'\b[a-fA-F0-9]{64}\b'), 'Indicator'),                      # SHA-256
#     (re.compile(r'\b[a-fA-F0-9]{40}\b'), 'Indicator'),                      # SHA-1
#     (re.compile(r'\b[a-fA-F0-9]{32}\b'), 'Indicator'),                      # MD5
#     (re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}(?::\d{1,5})?\b'), 'Indicator'),  # IPv4
#     (re.compile(r'\b(?:[A-Fa-f0-9]{1,4}:){2,7}[A-Fa-f0-9]{1,4}\b'), 'Indicator'),  # IPv6
#     (re.compile(r'\bhttps?://[^\s"<>]+', re.IGNORECASE), 'Indicator'),      # URL
#     (re.compile(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'), 'Indicator'),  # email
#     (re.compile(r'(?:/[A-Za-z0-9._\-]+){2,}'), 'System'),                   # POSIX path
#     (re.compile(r'\b[A-Z]:\\\\(?:[^\\\\ \\s"]+\\\\?)+', re.IGNORECASE), 'System'),  # Windows path
#     (re.compile(r'\b[A-Za-z0-9_-]+\.(?:exe|dll|so|jar|sh|bat|ps1|py)\b', re.IGNORECASE), 'System'),
# ]
# _DOMAIN_STOPWORDS = {'java.io', 'java.lang', 'java.util', 'java.net', 'sun.misc'}
# _DOMAIN_RE = re.compile(r'\b(?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.)+[a-z]{2,}\b', re.IGNORECASE)
#
#
# def find_spans(text: str) -> list[tuple[int, int, str]]:
#     spans: list[tuple[int, int, str]] = []
#     for pat, ent in PATTERNS:
#         for m in pat.finditer(text):
#             spans.append((m.start(), m.end(), ent))
#     for m in _DOMAIN_RE.finditer(text):
#         token = m.group(0).lower()
#         if token in _DOMAIN_STOPWORDS or any(token.startswith(p) for p in _DOMAIN_STOPWORDS):
#             continue
#         spans.append((m.start(), m.end(), 'Indicator'))
#     if not spans:
#         return []
#     spans.sort(key=lambda s: (s[0], -(s[1] - s[0])))
#     keep: list[tuple[int, int, str]] = []
#     last_end = -1
#     for s, e, ent in spans:
#         if s >= last_end:
#             keep.append((s, e, ent))
#             last_end = e
#     return keep
#
# if USE_REGEX_AUGMENTATION:
#     preview_texts, _, _ = build_windows(sample_normal=1, sample_failure=1, random_state=42)
#     for i, t in enumerate(preview_texts):
#         spans = find_spans(t)
#         print(f'\n--- preview {i}, {len(spans)} weak labels ---')
#         for s, e, ent in spans[:12]:
#             print(f'  [{ent:13}] "{t[s:e]}"')
#         if len(spans) > 12:
#             print(f'  ... [+{len(spans) - 12} more]')
# else:
#     print('Regex augmentation disabled.')


## 5. Sampled NER training run (verify environment first)

Trains on a ~10% sample of the CyNER train split (using the full `valid` split for evaluation) for two epochs of token-classification fine-tuning on `cisco-ai/SecureBERT2.0-NER`. Reports both overall and per-class seqeval F1 — pay attention to `Vulnerability` F1 given its low support. Output: `models/ner-sample/`.


In [ ]:
import random
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

# To consume the MLM-adapted base produced by kaggle_pretrain_mlm.ipynb,
# replace MODEL_ID with 'models/securebert2-logs-mlm/final' and reinitialise
# the classification head (HF will warn that the head weights are not loaded;
# that is intended).
MODEL_ID = 'cisco-ai/SecureBERT2.0-NER'

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print('CUDA:', torch.cuda.is_available(), 'bf16:', use_bf16, 'fp16:', use_fp16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# Fixture test: verify tokenize_and_align_labels with the loaded tokenizer
_fixture_enc = tokenize_and_align_labels(
    [{'tokens': FIXTURE_TOKENS, 'tags': FIXTURE_TAGS}], tokenizer
)[0]
assert len(_fixture_enc['input_ids']) == len(_fixture_enc['labels']), \
    'input_ids / labels length mismatch'
assert -100 in _fixture_enc['labels'], 'Expected -100 for special tokens'
print('Fixture tokenization OK:', len(_fixture_enc['input_ids']), 'subword tokens')
del _fixture_enc

# --- Sampled run: ~10% of CyNER train split ---
_rng = random.Random(42)
sample_size = max(50, len(data['train']) // 10)
sample_train = _rng.sample(data['train'], sample_size)
encoded_train = tokenize_and_align_labels(sample_train, tokenizer)
encoded_eval = tokenize_and_align_labels(data['valid'], tokenizer)
ds_train = Dataset.from_list(encoded_train)
ds_eval = Dataset.from_list(encoded_eval)
print(f'Sampled train: {len(ds_train)} sentences | eval (full valid): {len(ds_eval)} sentences')

# Label distribution sanity check
from collections import Counter as _Counter
label_counts = _Counter()
for ex in encoded_train:
    label_counts.update(l for l in ex['labels'] if l != -100)
print('Label counts (sampled train):')
for k, v in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {ID2LABEL[k]}: {v}')

collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

import evaluate as _evaluate
try:
    metric = _evaluate.load('seqeval')
except Exception:
    metric = None


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=-1)
    true_labels = [
        [ID2LABEL[l] for (p, l) in zip(pp, ll) if l != -100]
        for pp, ll in zip(preds, labels)
    ]
    true_preds = [
        [ID2LABEL[p] for (p, l) in zip(pp, ll) if l != -100]
        for pp, ll in zip(preds, labels)
    ]
    if metric is None:
        return {}
    res = metric.compute(predictions=true_preds, references=true_labels, zero_division=0)
    out = {
        'precision': res['overall_precision'],
        'recall': res['overall_recall'],
        'f1': res['overall_f1'],
        'accuracy': res['overall_accuracy'],
    }
    # Per-class F1 (critical for minority class Vulnerability)
    for ent_type in ['Indicator', 'Malware', 'Organization', 'System', 'Vulnerability']:
        if ent_type in res:
            out[f'f1_{ent_type}'] = res[ent_type]['f1']
            if ent_type == 'Vulnerability':
                n = res[ent_type].get('number', '?')
                print(f'  Vulnerability F1: {res["Vulnerability"]["f1"]:.4f}  (support={n})')
    return out


# OPTIONAL: class-weighted loss to compensate for Vulnerability imbalance.
# Uncomment WeightedNERTrainer and replace Trainer -> WeightedNERTrainer below.
#
# import torch.nn as nn
# class WeightedNERTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         labels = inputs.pop('labels')
#         outputs = model(**inputs)
#         logits = outputs.logits
#         weights = torch.ones(len(LABEL_LIST), device=logits.device)
#         for idx in [LABEL2ID['B-Vulnerability'], LABEL2ID['I-Vulnerability']]:
#             weights[idx] = 10.0  # up-weight minority class
#         loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
#         loss = loss_fn(logits.view(-1, len(LABEL_LIST)), labels.view(-1))
#         return (loss, outputs) if return_outputs else loss

sample_out = REPO_DIR / 'models' / 'ner-sample'
args = TrainingArguments(
    output_dir=str(sample_out),
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=use_bf16,
    fp16=use_fp16,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=50,
    save_total_limit=1,
    report_to='none',
    dataloader_num_workers=2,
    seed=42,
    metric_for_best_model='f1',
    load_best_model_at_end=True,
)
trainer = Trainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate()
print('sampled eval:', metrics)

trainer.save_model(str(sample_out / 'final'))
tokenizer.save_pretrained(str(sample_out / 'final'))
(sample_out / 'final' / 'ner_metrics.json').write_text(json.dumps(metrics, indent=2))
(sample_out / 'final' / 'ner_label_map.json').write_text(json.dumps(ID2LABEL, indent=2))
print('saved:', sample_out / 'final')


## 6. Full NER training run (uncomment when sampled run succeeds)

Trains on the complete CyNER train split (~2,811 sentences, ~68K tokens) for three epochs. Output: `models/ner/final/`. Estimated runtime ~30-60 min on Kaggle T4 with bf16/fp16.


In [ ]:
# Full NER training run on complete CyNER train split (~2,811 sentences, ~68K tokens).
encoded_train_full = tokenize_and_align_labels(data['train'], tokenizer)
encoded_eval_full = tokenize_and_align_labels(data['valid'], tokenizer)
ds_train_full = Dataset.from_list(encoded_train_full)
ds_eval_full = Dataset.from_list(encoded_eval_full)
print(f'Full train: {len(ds_train_full)} | eval: {len(ds_eval_full)}')

full_out = REPO_DIR / 'models' / 'ner'
args_full = TrainingArguments(
    output_dir=str(full_out),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=use_bf16,
    fp16=use_fp16,
    save_strategy='steps',
    save_steps=200,
    eval_strategy='steps',
    eval_steps=200,
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
    dataloader_num_workers=2,
    seed=42,
    metric_for_best_model='f1',
    load_best_model_at_end=True,
)
trainer_full = Trainer(
    model=AutoModelForTokenClassification.from_pretrained(
        MODEL_ID, num_labels=len(LABEL_LIST), id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    ),
    args=args_full,
    data_collator=collator,
    train_dataset=ds_train_full,
    eval_dataset=ds_eval_full,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer_full.train()
full_metrics = trainer_full.evaluate()
print('full eval:', full_metrics)
trainer_full.save_model(str(full_out / 'final'))
tokenizer.save_pretrained(str(full_out / 'final'))
(full_out / 'final' / 'ner_metrics.json').write_text(json.dumps(full_metrics, indent=2))
(full_out / 'final' / 'ner_label_map.json').write_text(json.dumps(ID2LABEL, indent=2))
print('full saved:', full_out / 'final')


## 7. Inspect artifacts and export to ONNX

ONNX export uses `optimum` so the same artifact format that production NER inference expects is produced here. The export is over the picked checkpoint (full > sample) — copy to `models/ner/final/` if you only ran the sampled cell.


In [ ]:
from optimum.onnxruntime import ORTModelForTokenClassification

full_dir = REPO_DIR / 'models' / 'ner' / 'final'
sample_dir = REPO_DIR / 'models' / 'ner-sample' / 'final'
src_dir = full_dir if full_dir.exists() else sample_dir

if not src_dir.exists():
    raise FileNotFoundError('No NER artifact found. Run the sampled or full training cell first.')

print(f'=== {src_dir} ===')
for path in sorted(src_dir.glob('*')):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size:,} bytes')

ort = ORTModelForTokenClassification.from_pretrained(src_dir, export=True)
ort.save_pretrained(src_dir)
print('ONNX exported into:', src_dir)
for path in sorted(src_dir.glob('*.onnx')):
    print(f'  {path.name}: {path.stat().st_size:,} bytes')


## 8. Package artifacts

Zip `models/ner/final/` into `/kaggle/working/` so it appears in Kaggle's notebook outputs.


In [ ]:
import shutil

tag = 'full' if (REPO_DIR / 'models' / 'ner' / 'final').exists() else 'sample'
src_dir = REPO_DIR / ('models/ner/final' if tag == 'full' else 'models/ner-sample/final')
out_zip = KAGGLE_WORKING / f'logfilter-ner-{tag}-artifacts'
shutil.make_archive(
    str(out_zip), 'zip',
    root_dir=REPO_DIR,
    base_dir=Path(*src_dir.parts[-3:]).as_posix(),
)
print('packaged:', out_zip.with_suffix('.zip'))


## 9. Output description + how to consume in repo

Download `/kaggle/working/logfilter-ner-full-artifacts.zip` (or `-sample-` for the verification run) and extract it at the repository root. The archive contains:

```text
models/ner/final/config.json
models/ner/final/model.safetensors
models/ner/final/tokenizer.json (and tokenizer_config.json, special_tokens_map.json)
models/ner/final/model.onnx
models/ner/final/ner_metrics.json         <- includes per-class F1 (f1_Indicator, f1_Vulnerability, ...)
models/ner/final/ner_label_map.json
```

To use this fine-tuned NER inside the production pipeline, point [src/logfilter/models/ner.py](../src/logfilter/models/ner.py) at the local directory:

```yaml
# config/config.yaml
models:
  ner:
    model_id: "models/ner/final"   # was: cisco-ai/SecureBERT2.0-NER
    device: "cpu"
    batch_size: 32
    min_confidence: 0.80
```

**Promotion gate — domain transfer eval.** Before promoting to production, run the trained model on 200 held-out syslog windows and compare entity yield against the baseline regex (`find_spans` in the optional Section 4 cell). Promote only if `Indicator` and `Vulnerability` F1 (or yield, if ground truth is unavailable) meets or exceeds the regex baseline on the syslog corpus:

```python
# from logfilter.models.ner import NERModel
# from training.text_dataset import build_windows
# ner = NERModel(model_id=str(REPO_DIR / 'models' / 'ner' / 'final'), device='cpu')
# eval_texts, _, _ = build_windows(sample_normal=100, sample_failure=100, random_state=99)
# results = ner.extract_batch(eval_texts[:200])
# n_indicators = sum(len(r.indicators) for r in results)
# n_vulns = sum(len(r.vulnerabilities) for r in results)
# print(f'NER yield on 200 syslog windows: {n_indicators} Indicator, {n_vulns} Vulnerability')
# # Compare to regex baseline:
# # regex_spans = [find_spans(t) for t in eval_texts[:200]]
# # n_regex_ind = sum(1 for spans in regex_spans for _, _, e in spans if e == 'Indicator')
# # print(f'Regex yield: {n_regex_ind} Indicator')
```

Caveats:

* **CyNER ≠ syslog.** CyNER is CTI prose; log syntax differs. Run the domain-transfer yield check above before production promotion. If recall on `Indicator` is materially lower than regex on the same syslog corpus, the fine-tune may have overfit to CTI surface forms.
* **Vulnerability imbalance.** CyNER contains only ~67 `B-Vulnerability` tokens. Treat Vulnerability F1 as an aspirational metric; the class-weighted loss option in Section 5 can partially compensate.
* **Per-class F1 in `ner_metrics.json`.** The output JSON includes `f1_Indicator`, `f1_Malware`, `f1_Organization`, `f1_System`, `f1_Vulnerability` keys in addition to overall F1.
* **Composability with MLM pre-training.** For best results, run [kaggle_pretrain_mlm.ipynb](kaggle_pretrain_mlm.ipynb) first, then change `MODEL_ID` to `models/securebert2-logs-mlm/final` for a log-adapted encoder before NER fine-tuning.
